In [ ]:
from collections import deque, defaultdict


knight_moves = [
    (2, 1), (2, -1), (-2, 1), (-2, -1),
    (1, 2), (1, -2), (-1, 2), (-1, -2)
]


TARGET_SCORE = 2024
ROWS, COLS = 6, 6

# Grid layout
grid = [
    ['A', 'B', 'B', 'C', 'C', 'C'],  # Row 6
    ['A', 'B', 'B', 'C', 'C', 'C'],  # Row 5
    ['A', 'A', 'B', 'B', 'C', 'C'],  # Row 4
    ['A', 'A', 'B', 'B', 'C', 'C'],  # Row 3
    ['A', 'A', 'A', 'B', 'B', 'C'],  # Row 2
    ['A', 'A', 'A', 'B', 'B', 'C']   # Row 1
]

# Calculate the new score after a move
def calculate_new_score(current_score, current_value, next_value):
    if current_value == next_value:
        return current_score + next_value  # Add if moving to the same integer type
    else:
        return current_score * next_value  # Multiply if moving to a different integer type

# DP-based pathfinding with knight's move and scoring constraints
def dp_pathfinding(start_x, start_y, target_x, target_y, target_score, values, penultimate_pos):
    dp = defaultdict(lambda: None)
    queue = deque([(start_x, start_y, values['A'], [(start_x, start_y)])])  # Start with the initial score set to A

    while queue:
        x, y, score, path = queue.popleft()

        # Check if reached the penultimate square and can complete the target score condition
        if (x, y) == penultimate_pos:
            next_score = calculate_new_score(score, values[grid[x][y]], values['C'])
            if next_score == target_score:
                return path + [(target_x, target_y)]

        # Explore all possible knight moves
        for dx, dy in knight_moves:
            nx, ny = x + dx, y + dy
            if 0 <= nx < ROWS and 0 <= ny < COLS and (nx, ny) not in path:
                current_value = values[grid[x][y]]
                next_value = values[grid[nx][ny]]
                new_score = calculate_new_score(score, current_value, next_value)

                if new_score <= target_score:
                    if dp[(nx, ny, new_score)] is None or len(dp[(nx, ny, new_score)]) > len(path) + 1:
                        dp[(nx, ny, new_score)] = path + [(nx, ny)]
                        queue.append((nx, ny, new_score, path + [(nx, ny)]))

    return None

# Find paths based on penultimate square conditions
def find_paths():
    min_sum = float('inf')
    best_values = None
    best_paths = None

    # Loop through possible values of A, B, and C within the specified range
    for Av in range(1, 30):
        for Bv in range(1, 30):
            for Cv in range(1, 30):
                if Av != Bv and Bv != Cv and Av != Cv and Av + Bv + Cv < min_sum:
                    values = {'A': Av, 'B': Bv, 'C': Cv}

                    # Path 1: From a1 to f6, ending at d5 or e4
                    path1 = dp_pathfinding(5, 0, 0, 5, TARGET_SCORE, values, (1, 3)) \
                            or dp_pathfinding(5, 0, 0, 5, TARGET_SCORE, values, (2, 4))

                    # Path 2: From a6 to f1, ending at d2 or e3
                    path2 = dp_pathfinding(0, 0, 5, 5, TARGET_SCORE, values, (4, 3)) \
                            or dp_pathfinding(0, 0, 5, 5, TARGET_SCORE, values, (3, 4))

                    # Check if both paths are valid
                    if path1 and path2:
                        total_sum = Av + Bv + Cv
                        if total_sum < min_sum:
                            min_sum = total_sum
                            best_values = (Av, Bv, Cv)
                            best_paths = (path1, path2)

    return best_values, best_paths

# Execute the function and print results
best_values, best_paths = find_paths()
if best_values:
    Av, Bv, Cv = best_values
    print(f"Minimum sum found: Av={Av}, Bv={Bv}, Cv={Cv} with sum {Av + Bv + Cv}")
    print("Path from a1 to f6:")
    values = {'A': Av, 'B': Bv, 'C': Cv}
    print(",".join([f"{chr(py + ord('a'))}{ROWS - px}" for px, py in best_paths[0]]))

    print("\nPath from a6 to f1:")
    print(",".join([f"{chr(py + ord('a'))}{ROWS - px}" for px, py in best_paths[1]]))
else:
    print("No valid solution found.")


Minimum sum found: Av=1, Bv=3, Cv=2 with sum 6
Path from a1 to f6:
a1,b3,c1,d3,e5,c4,d6,e4,f2,d1,b2,a4,b6,d5,f6

Path from a6 to f1:
a6,b4,c2,d4,e6,c5,b3,c1,d3,e1,f3,d2,c4,e3,f1
